# On-device continual triage with **online SDFT** (LFM2.5-230M) — 3-way, 3 regimes

A phone-hosted 230M model learns your **drifting** attention policy — triage each
notification as **INTERRUPT** / **LATER** / **ARCHIVE** — from **implicit feedback**
(open now / let it wait / never opened), **no gold labels**, one `batch_size=1` LoRA
step at a time. The learned policy lives in a **~1.4 MB adapter**, so serving is a
bare prompt: no growing in-context cheat-sheet (ICL), no retrieval index (RAG).

**Two drifts, three regimes across a realistic week** — 50% regular **weekdays**,
20% **on-call**, 30% **off-hours** (evenings + the weekend), streamed
`weekday → on-call → off-hours`. Several categories flip on each drift — monitoring
alerts `ARCHIVE → INTERRUPT` when you go on-call; manager pings
`INTERRUPT → LATER → ARCHIVE` across all three; social
`ARCHIVE → ARCHIVE → INTERRUPT` once you're off the clock. The weights track whichever
regime you're living in: on our seeded MPS run each regime peaks *inside its own
window* (weekday 1.00, on-call 0.92, off-hours 0.83), and the live stream ends at
**18/60 mistakes — vs 30+ for every frozen baseline** (accumulated regret, Panel C).

**3-way is genuinely harder than binary** on a 230M — `LATER` and `ARCHIVE` both mean
"don't buzz me", so they partly conflate. Two tricks matter: every `batch_size=1`
update replays one item from *each* other class, and because 3-way can *over-train
and decay* past its peak we keep the best adapter on the current policy — the same
**probe guardrail / auto-rollback** the repo's `sdft.online` package ships. Every
method is *causal*, like a real device: ICL pins the most recent observed
decisions, RAG retrieves only from decisions already made, and SDFT guesses with
the live adapter before each label lands. ICL and RAG go straight in at their
sweep-winning context size k — the sweep itself lives in the repo's
`run_baselines.py`, so the notebook spends your Colab GPU quota on the
part only it can show: the online learning.

**Standalone** — no `git clone`; the one input it needs, the seeded dataset the
repo's scripts trained on, is fetched straight from the repo at start. Paper:
[Shenfeld et al., 2026](https://self-distillation.github.io/SDFT)
([arXiv](https://arxiv.org/abs/2601.19897)). Base:
[Liquid AI LFM2.5-230M](https://huggingface.co/LiquidAI/LFM2.5-230M).


## Setup
Colab: **Runtime → GPU** (T4 is plenty). Installs deps and drops stale `torchao`.

In [ ]:
%pip install -q "transformers>=4.54" "trl>=0.19" "peft>=0.15" "datasets>=2.19" "accelerate>=0.33" matplotlib
%pip uninstall -y -q torchao
print("setup ok — 3-way / 3-regime online SDFT triage demo")

## Fetch the seeded dataset (the same one the repo's scripts trained on)

One `urllib` call replaces ~100 lines of generator code: the repo ships
[`data/inbox_triage.json`](https://github.com/lin826/online-sdft-triage/blob/main/data/inbox_triage.json)
— the 60-item drifting stream, the three 12-item held-out eval sets, and the
config, every item with its prompt pre-rendered. The labels encode this latent
policy (bold = a drift flip):

| Category        | weekday   | on-call       | off-hours     |
| --------------- | --------- | ------------- | ------------- |
| `mgr_project`   | INTERRUPT | **LATER**     | **ARCHIVE**   |
| `calendar_soon` | INTERRUPT | INTERRUPT     | **LATER**     |
| `teammate_fyi`  | LATER     | **ARCHIVE**   | ARCHIVE       |
| `monitoring`    | ARCHIVE   | **INTERRUPT** | **ARCHIVE**   |
| `promo`         | ARCHIVE   | ARCHIVE       | **LATER**     |
| `social`        | ARCHIVE   | ARCHIVE       | **INTERRUPT** |
| `receipt`       | LATER     | **ARCHIVE**   | **LATER**     |

Blocks are class-balanced under their own policy (10/10/10, 4/4/4, 6/6/6) across
a realistic 50/20/30 week. Items are deliberately bare — two lines, subject +
snippet, no channel field, no role tags — and the vocabulary sets a trap:
`receipt` ("your **payment** succeeded") and `monitoring` ("**payment** latency")
share words but land on different rungs — you must learn the *policy*, not the
keywords. (The generator lives in `triage_common.py` in the repo.)

In [ ]:
import copy, json, random, re, urllib.request
import torch
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict, set_peft_model_state_dict
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "LiquidAI/LFM2.5-230M"
DATA_URL = "https://raw.githubusercontent.com/lin826/online-sdft-triage/main/data/inbox_triage.json"

with urllib.request.urlopen(DATA_URL) as fh:
    data = json.load(fh)
cfg = data["config"]
SEED = cfg["seed"]
ACTIONS = tuple(cfg["actions"])         # INTERRUPT / LATER / ARCHIVE
REGIMES = tuple(cfg["regimes"])         # weekday / on-call / off-hours
STREAM_LEN = cfg["stream_len"]
DRIFTS = tuple(cfg["drifts"])           # weekday items 1-30 | on-call 31-42 | off-hours 43-60

CHECKPOINTS = (6, 12, 18, 24, 30, 36, 42, 48, 54, 60)
LORA_R, LORA_ALPHA = 8, 16   # sweep winner (sweep_sdft.py) — ~1.4 MB adapter
LORA_TARGET = r".*self_attn\.(q|k|v|out)_proj"
LR, MAX_NEW = 7e-4, 8   # 1e-3 overshoots on the live stream (regret 28 vs 18)
ICL_K = 12  # ICL / RAG served at their sweep-winning context size, judged on the
RAG_K = 6   # whole-week mean; the sweep itself runs locally in run_baselines.py
ICL_K_REGRET = 12   # Panel C serves each baseline at its LEAST-REGRET k instead —
RAG_K_REGRET = 3    # swept separately (most flattering setting per metric)
REPLAY, STEPS_PER_ITEM = 16, 8   # regret-primary sweep winners (sweep_sdft.py)

stream = data["stream"]
evals = {int(k): v for k, v in data["evals"].items()}
eval_cur = evals[3]   # the current policy is the final regime = off-hours
print(f"fetched {len(stream)} stream + {sum(map(len, evals.values()))} eval items; regimes {REGIMES}")

## Model + pipeline helpers (inlined)

In [ ]:
def device():
    if torch.cuda.is_available(): return "cuda"
    if getattr(torch.backends,"mps",None) and torch.backends.mps.is_available(): return "mps"
    return "cpu"
DEV = device(); print("device:", DEV)

def load_tok():
    t = AutoTokenizer.from_pretrained(MODEL_NAME)
    if t.pad_token is None: t.pad_token = t.eos_token
    t.padding_side = "left"; return t
def load_base():
    dt = torch.float16 if DEV=="cuda" else torch.float32
    m = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dt)
    return m if DEV=="cuda" else m.to(DEV)
def to_dev(enc, model):
    d = next(model.parameters()).device
    return {k:(v.to(d) if torch.is_tensor(v) else v) for k,v in enc.items()}

def demo_msg(it, a): return [{"role":"user","content":it["prompt"]},{"role":"assistant","content":a}]
def build_msgs(it, demos=None):
    m=[]
    for d,a in demos or []: m += demo_msg(d,a)
    m.append({"role":"user","content":it["prompt"]}); return m

ACTION_RE = re.compile(r"\b(INTERRUPT|LATER|ARCHIVE)\b", re.I)
def parse_action(t):
    m = ACTION_RE.search(t or ""); return m.group(1).upper() if m else "NONE"
def prompt_tokens(tok, msgs):
    txt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return len(tok(txt, add_special_tokens=False)["input_ids"])
def accuracy(items, gens):
    return sum(parse_action(g)==it["action"] for it,g in zip(items,gens))/max(len(items),1)

@torch.inference_mode()
def generate(model, tok, items, demos_fn, bs=8, max_new=MAX_NEW):
    model.eval(); outs=[]
    for s in range(0,len(items),bs):
        batch=items[s:s+bs]
        txts=[tok.apply_chat_template(demos_fn(it),tokenize=False,add_generation_prompt=True) for it in batch]
        enc=to_dev(tok(txts,return_tensors="pt",padding=True,add_special_tokens=False),model)
        out=model.generate(**enc,max_new_tokens=max_new,do_sample=False,pad_token_id=tok.pad_token_id)
        for t in tok.batch_decode(out[:,enc["input_ids"].shape[1]:],skip_special_tokens=True):
            outs.append(t.strip())
    return outs

def retriever(store):
    toks=[set(re.findall(r"\w+",s["prompt"].lower())) for s in store]
    def r(it,k,upto=None):   # upto = only the first `upto` decisions exist yet (causal)
        pool=range(len(store) if upto is None else min(upto,len(store)))
        q=set(re.findall(r"\w+",it["prompt"].lower()))
        order=sorted(pool,key=lambda i:-len(q&toks[i]))
        return [(store[i],store[i]["action"]) for i in order[:k]]
    return r

def phase_of(pos):   # regime of stream position pos; 0 (nothing yet) = upcoming weekday
    return 1 if pos<=DRIFTS[0] else 2 if pos<=DRIFTS[1] else 3
def recent_demos(history,k):   # the k most recent observed decisions, oldest first
    return [(it,it["action"]) for it in history[-k:]]

# persistent-optimizer online updater (Adam momentum across the stream) + grad clipping
def make_updater(model, tok):
    trainable = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(trainable, lr=LR)
    eos = tok.eos_token or ""
    dev = next(model.parameters()).device
    def step(rows, steps):
        model.train(); model.config.use_cache = False
        for k in range(steps):
            r = rows[k % len(rows)]
            ptxt = tok.apply_chat_template([{"role":"user","content":r["prompt"]}],
                                           tokenize=False, add_generation_prompt=True)
            ids_p = tok(ptxt, add_special_tokens=False)["input_ids"]
            ids_c = tok(r["target"] + eos, add_special_tokens=False)["input_ids"]
            ids = torch.tensor([ids_p + ids_c], device=dev)
            labels = torch.tensor([[-100]*len(ids_p) + ids_c], device=dev)  # completion-only loss
            model(input_ids=ids, labels=labels).loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)                  # keep bs=1 steps stable
            opt.step(); opt.zero_grad()
        model.config.use_cache = True; model.eval()
    return step
print("helpers ready")

## A look at the fetched stream

In [ ]:
from collections import Counter
for p in (1,2,3):
    print(f"phase {p} ({REGIMES[p-1]}):", dict(Counter(it["action"] for it in stream if it["phase"]==p)))
def flip(cat):   # how one category's label moves across the three regimes
    return [next(it["action"] for it in stream if it["phase"]==p and it["category"]==cat) for p in (1,2,3)]
print("\ntriple-flip — SAME manager ping, three regimes:", flip("mgr_project"))
print("social flips ARCHIVE->ARCHIVE->INTERRUPT:", flip("social"))
print("\nsample item:\n", stream[0]["prompt"][:230])

## Causal baselines, graded on the whole week

Same information diet as the learner: at stream position t, ICL pins the **k most
recent** observed decisions in context and RAG retrieves k neighbours from the
**decisions observed so far** — no future items, no hand-curated cheat-sheet.
(Their k comes from the sweep in the repo's `run_baselines.py`, judged on
the whole-week mean.) Frozen context has to pick a lane: ICL's recent window
chases the newest regime, RAG's store stays dominated by the weekday bulk —
neither serves all three policies at once.

In [ ]:
tok = load_tok(); base = load_base()
retrieve = retriever(stream)
def icl_ctx(it, upto): return recent_demos(stream[:upto], ICL_K)
def rag_ctx(it, upto): return retrieve(it, RAG_K, upto=upto)

def arm_eval(ctx_fn):   # whole-week arm: mean accuracy + token bill over ALL 3 regime evals
    accs, bills = {}, []
    for p in (1,2,3):
        msgs = lambda it: build_msgs(it, ctx_fn(it, STREAM_LEN) if ctx_fn else None)
        accs[p] = accuracy(evals[p], generate(base,tok,evals[p],msgs,bs=8))
        bills += [prompt_tokens(tok,msgs(it)) for it in evals[p]]
    return dict(by_phase=accs, mean=sum(accs.values())/3, tok=sum(bills)/len(bills))

arms = {"ZS": arm_eval(None), f"ICL k={ICL_K}": arm_eval(icl_ctx), f"RAG k={RAG_K}": arm_eval(rag_ctx)}
for n,a in arms.items():
    print(f"  {n:9s} mean={a['mean']:.2f}  " +
          " ".join(f"{REGIMES[p-1]}={a['by_phase'][p]:.2f}" for p in (1,2,3)) + f"  tok/q={a['tok']:.0f}")

## The baselines along the stream: checkpoint curves + accumulated regret

Two online views, both causal. **Curves**: at each checkpoint, accuracy on the
*current* regime's held-out set with only the history available at that point —
position 0 (nothing streamed) is the zero-shot value for every method.
**Regret**: predict each streamed item *before* its label lands and count
cumulative mistakes — the scoreboard an online-RL eye goes to first. Here ICL
and RAG run at their **least-regret** k (swept locally, like the accuracy k) —
each frozen baseline at its most flattering setting for this metric too.

In [ ]:
zs_by_phase = arms["ZS"]["by_phase"]
curves = {"pos": [0, *CHECKPOINTS], "ICL": [zs_by_phase[1]], "RAG": [zs_by_phase[1]]}
for pos in CHECKPOINTS:
    p = phase_of(pos)
    curves["ICL"].append(accuracy(evals[p], generate(base,tok,evals[p],lambda it: build_msgs(it, icl_ctx(it,pos)),bs=8)))
    curves["RAG"].append(accuracy(evals[p], generate(base,tok,evals[p],lambda it: build_msgs(it, rag_ctx(it,pos)),bs=8)))
    print(f"  pos {pos:2d} ({REGIMES[p-1]}): ICL={curves['ICL'][-1]:.2f}  RAG={curves['RAG'][-1]:.2f}")

def icl_ctx_r(it, upto): return recent_demos(stream[:upto], ICL_K_REGRET)
def rag_ctx_r(it, upto): return retrieve(it, RAG_K_REGRET, upto=upto)
regret = {"pos": list(range(STREAM_LEN+1))}
for name, ctx in (("ZS", None), ("ICL", icl_ctx_r), ("RAG", rag_ctx_r)):
    preds = [parse_action(g) for g in generate(base,tok,list(range(STREAM_LEN)),
             lambda i: build_msgs(stream[i], ctx(stream[i], i) if ctx else None), bs=8)]
    c = [0]
    for pr,it in zip(preds,stream): c.append(c[-1] + int(pr != it["action"]))
    regret[name] = c
    print(f"  regret {name}: {c[-1]}/{STREAM_LEN} stream mistakes")

## The online SDFT loop: guess → feedback → `batch_size=1` update → guardrail

Prequential, like a real device: on every incoming item the **current adapter**
makes the call first — the exact bare prompt it serves, and that prediction
feeds the regret curve *before* the label lands — then your observed behaviour
reinforces it when right, corrects it when wrong. The target is always a bare
action, never a hand-written gold answer. `torch.manual_seed` makes the LoRA init
(and so the whole run) repeatable. At each checkpoint we probe all three regime
policies; during the final regime the guardrail snapshots the best off-hours
adapter and rolls back to it before serving.

In [ ]:
torch.manual_seed(SEED)   # LoRA init + dropout masks — makes the run repeatable
model = get_peft_model(base, LoraConfig(r=LORA_R,lora_alpha=LORA_ALPHA,lora_dropout=0.05,
                                        target_modules=LORA_TARGET,task_type="CAUSAL_LM"))
update = make_updater(model, tok)
curve = {"pos":[], 1:[], 2:[], 3:[]}
best = {"acc":-1.0, "pos":None, "state":None}
buf=[]; rr=random.Random(SEED); reinforced=0; sdft_regret=[0]
for i,it in enumerate(stream):
    # the model's OWN call — the exact bare prompt it serves, scored before the label lands
    g = generate(model,tok,[it],lambda x: build_msgs(x),bs=1)[0]
    fb = int(parse_action(g)==it["action"])   # 1 = reinforce its own answer, 0 = correct toward yours
    reinforced += fb; sdft_regret.append(sdft_regret[-1] + 1 - fb)
    row = {"prompt":it["prompt"],"target":it["action"]}
    buf=(buf+[row])[-REPLAY:]
    batch=[row]   # pair the fresh item with one replayed item from EACH other class
    for a in ACTIONS:
        pool=[b for b in buf[:-1] if b["target"]==a]
        if a!=it["action"] and pool: batch.append(rr.sample(pool,1)[0])
    update(batch, STEPS_PER_ITEM)
    if (i+1) in CHECKPOINTS:
        curve["pos"].append(i+1)
        for p in (1,2,3):
            curve[p].append(accuracy(evals[p], generate(model,tok,evals[p],lambda x:build_msgs(x),bs=8)))
        if (i+1) > DRIFTS[1] and curve[3][-1] >= best["acc"]:      # off-hours window: keep best-on-current
            best = {"acc":curve[3][-1], "pos":i+1, "state":copy.deepcopy(get_peft_model_state_dict(model))}
        print(f"  streamed {i+1:2d}: " + " ".join(f"{REGIMES[p-1]}={curve[p][-1]:.2f}" for p in (1,2,3))
              + f"  (stream mistakes {sdft_regret[-1]}/{i+1})")
print(f"reinforced online (model already right): {reinforced}/{len(stream)}")
if best["state"] is not None:                              # auto-rollback to the best off-hours adapter
    set_peft_model_state_dict(model, best["state"])
    print(f"probe guardrail: serving the adapter from item {best['pos']} (off-hours acc {best['acc']:.2f})")
served = {p: accuracy(evals[p], generate(model,tok,evals[p],lambda x:build_msgs(x),bs=8)) for p in (1,2,3)}
arms["Online-SDFT"] = dict(by_phase=served, mean=sum(served.values())/3, tok=arms["ZS"]["tok"])
curves["SDFT"] = [zs_by_phase[1]] + [curve[phase_of(pos)][idx] for idx,pos in enumerate(curve["pos"])]
regret["SDFT"] = sdft_regret

# Panel A grades the week AS LIVED: each regime scored while it is live (its
# block-end checkpoint, identical for every method), averaged. arms[n]["mean"]
# keeps the week-END re-grade — tiny adapters forget, and that number shows it.
block_ends = [*DRIFTS, STREAM_LEN]
for n,key in (("ZS","ZS"),(f"ICL k={ICL_K}","ICL"),(f"RAG k={RAG_K}","RAG"),("Online-SDFT","SDFT")):
    live = {p:(zs_by_phase[p] if key=="ZS" else curves[key][curves["pos"].index(pos)])
            for pos,p in zip(block_ends,(1,2,3))}
    arms[n]["mean_live"] = sum(live.values())/3
print("whole-week (each regime while live):", {n:round(a["mean_live"],2) for n,a in arms.items()})

## The figure: one adapter vs frozen context, three ways

**A** grades the *whole week* — each regime scored while it is live (its
block-end checkpoint, identical for every method), averaged — against the
tokens paid per query. **B** follows the same methods along the stream on the
*current* regime's held-out policy: everyone starts at the zero-shot point,
everyone gets the same causal history. **C** is the online-learning
scoreboard: cumulative mistakes on the streamed items themselves.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.transforms import blended_transform_factory
colors={"ZS":"#9aa0a6","ICL":"#e8710a","RAG":"#d93025","Online-SDFT":"#1a73e8"}
def col(n):
    for k,c in colors.items():
        if n.startswith(k): return c
    return "#5f6368"
zs_tok = arms["ZS"]["tok"]
bounds = [0, *DRIFTS, STREAM_LEN]
fig,(axA,axB,axC)=plt.subplots(1,3,figsize=(18.4,4.9))

# A — whole-week accuracy (each regime while live) vs the recurring token bill
for n,d in arms.items():
    axA.scatter(d["tok"],d["mean_live"]*100,s=170,color=col(n),zorder=3,edgecolor="white",linewidth=1.5)
    axA.annotate(n,(d["tok"],d["mean_live"]*100),textcoords="offset points",
                 xytext=(8, -12 if n.startswith("RAG") else 4),fontsize=10.5,fontweight="bold",color=col(n))
axA.set_xlabel("Recurring prompt tokens / query  (paid on every notification)")
axA.set_ylabel("Whole-week accuracy: mean over regimes,\neach scored while live (%)")
axA.set_ylim(0,105); axA.grid(alpha=.25)
axA.axvspan(0,zs_tok+22,color="#1a73e8",alpha=.05)
axA.text((zs_tok+22)/2,6,"bare-prompt zone\n(weights carry the policy)",ha="center",fontsize=8.5,color="#1a73e8",style="italic")
axA.set_title("A.  Whole-week accuracy vs the recurring token bill",fontweight="bold")

# B — current-regime curves, every method from the same zero-shot start
for x in DRIFTS: axB.axvline(x,color="#5f6368",ls="--",lw=1.2)
for (s,e),p in zip(zip(bounds,bounds[1:]),(1,2,3)):
    axB.hlines(zs_by_phase[p]*100,s,e,color=colors["ZS"],ls=":",lw=1.6,zorder=1)
for key,n in (("SDFT","Online-SDFT"),("ICL",f"ICL k={ICL_K}"),("RAG",f"RAG k={RAG_K}")):
    axB.plot(curves["pos"],[v*100 for v in curves[key]],"-o",color=colors["Online-SDFT" if key=="SDFT" else key],
             lw=2.6 if key=="SDFT" else 1.8,ms=5 if key=="SDFT" else 3.8,zorder=4 if key=="SDFT" else 3)
if best["pos"]:
    axB.scatter([best["pos"]],[best["acc"]*100],marker="*",s=300,color=colors["Online-SDFT"],
                edgecolor="white",linewidth=1.2,zorder=5)
    axB.annotate("probe keeps\nthis adapter",(best["pos"],best["acc"]*100),textcoords="offset points",
                 xytext=(0,-24 if best["acc"]>0.88 else 10),fontsize=7.5,
                 color=colors["Online-SDFT"],ha="center",fontweight="bold")
tr = blended_transform_factory(axB.transData, axB.transAxes)
for idx,(s,e,r) in enumerate(zip(bounds,bounds[1:],REGIMES)):
    if idx%2: axB.axvspan(s,e,color="#5f6368",alpha=.05,zorder=0)
    axB.text((s+e)/2,-0.115,f"{r}\nitems {s+1}–{e}",transform=tr,ha="center",va="top",
             fontsize=8.3,color="#5f6368",fontweight="bold")
axB.set_xlim(-1,STREAM_LEN+2); axB.set_ylim(0,105); axB.grid(alpha=.25)
axB.set_xlabel("Items streamed  (same causal history for every method)",labelpad=36)
axB.set_ylabel("Held-out accuracy on the CURRENT regime (%)")
axB.set_title("B.  Who tracks the drifting policy",fontweight="bold")

# C — accumulated regret: cumulative mistakes on the streamed items
for x in DRIFTS: axC.axvline(x,color="#5f6368",ls="--",lw=1.2)
for key,n,ls in (("ZS","ZS",":"),("ICL",f"ICL k={ICL_K_REGRET}","-"),("RAG",f"RAG k={RAG_K_REGRET}","-"),("SDFT","Online-SDFT","-")):
    c=colors["Online-SDFT" if key=="SDFT" else key]
    axC.plot(regret["pos"],regret[key],ls,drawstyle="steps-post",color=c,
             lw=2.6 if key=="SDFT" else 1.8,label=n,zorder=4 if key=="SDFT" else 3)
    axC.annotate(str(regret[key][-1]),(STREAM_LEN,regret[key][-1]),textcoords="offset points",
                 xytext=(6,-3),fontsize=9,color=c,fontweight="bold")
axC.set_xlim(-1,STREAM_LEN+4); axC.grid(alpha=.25)
axC.set_xlabel("Items streamed"); axC.set_ylabel("Cumulative mistakes on streamed items")
axC.set_title("C.  Accumulated regret — baselines at their least-regret k",fontweight="bold")
axC.legend(fontsize=8.5,loc="upper left",framealpha=.95)
fig.suptitle("On-device 3-way triage across 3 regimes · LFM2.5-230M · policy in a ~1.4 MB LoRA adapter, no gold labels",
             fontweight="bold",y=1.02)
fig.tight_layout(); plt.show()

## One item, four minds
An off-hours **social** push — off the clock it flips to **INTERRUPT** ("your
friends' posts DO deserve a buzz on Saturday"), where the base model files everything
as `LATER`. We show the first such item the served adapter gets right.

In [ ]:
socials = [it for it in eval_cur if it["category"]=="social"]
def act(adapter_off, demos, it):
    if adapter_off:
        with model.disable_adapter():
            return parse_action(generate(model,tok,[it],lambda x:build_msgs(x,demos),bs=1)[0])
    return parse_action(generate(model,tok,[it],lambda x:build_msgs(x,demos),bs=1)[0])
# pick a social item the served adapter gets right but the frozen base doesn't
q = next((it for it in socials if act(False,None,it)==it["action"] and act(True,None,it)!=it["action"]), socials[0])
print("PROMPT:\n", q["prompt"], "\n\ncorrect (off-hours):", q["action"], "\n")
extra = arms[f"ICL k={ICL_K}"]["tok"] - arms["ZS"]["tok"]
print("  ZS          ->", act(True, None, q))
print(f"  ICL k={ICL_K}     ->", act(True, recent_demos(stream,ICL_K), q), f"  (+{extra:.0f} tokens every call)")
print(f"  RAG k={RAG_K}     ->", act(True, retrieve(q,RAG_K), q))
print("  Online-SDFT ->", act(False, None, q), "  (bare prompt)")

## What to try next
1. **Change the policy (or make regimes longer)** — the generator lives in [triage_common.py](https://github.com/lin826/Local-SDFT/blob/main/triage_common.py); edit `POLICY` or the phase specs there, re-run `run_baselines.py` to re-export `data/inbox_triage.json`, and point `DATA_URL` at your fork.
2. **Noisier feedback** — flip a fraction of observed actions before training; watch the guardrail earn its keep.
3. **Re-run the k sweep locally** — `run_baselines.py` sweeps k ∈ {3, 6, 9, 12} on your machine (widen it!); on Colab just try other `ICL_K` / `RAG_K` values. Does RAG ever stop serving stale weekday answers?
4. **Bigger model** — swap `MODEL_NAME` to `LiquidAI/LFM2.5-1.2B-Instruct`; 3-way routing is much steadier with more capacity.
5. **Full toolkit** — [Local-SDFT](https://github.com/lin826/Local-SDFT): the `sdft.online` serve-while-learning package (the real probe guardrail + auto-rollback), the `/data` live chat loop, and GRPO.

3-way on a 230M is at the edge — that's the fun. Curiosity, not VRAM, is the bottleneck.